In [ ]:
RUN_TARGET = "local"  # "colab" | "local"


## Setup Instructions

Notebook 07 visualizes and summarizes saved probe-row CSVs only. It does not load models or generate attributions. The workflow is semi-automatic over the supported registry, not fully generic over arbitrary files in `models/` or `results/`. Main-paper outputs use a curated subset of models and signals; supplementary outputs retain the fuller active-method inventory.


In [ ]:
if RUN_TARGET == "colab":
    import importlib.metadata as _md
    import subprocess as _sp
    import sys as _sys

    _required = {
        "numpy": "1.26.4",
        "scipy": "1.15.3",
        "scikit-learn": "1.8.0",
        "transformers": "4.48.1",
        "huggingface-hub": "0.36.2",
        "statsmodels": "0.14.5",
        "seaborn": "0.13.2",
    }

    def _version_matches(name: str, expected: str) -> bool:
        try:
            return _md.version(name) == expected
        except _md.PackageNotFoundError:
            return False

    _missing_or_mismatched = [
        f"{name}=={version}"
        for name, version in _required.items()
        if not _version_matches(name, version)
    ]

    if _missing_or_mismatched:
        print("Installing:", ", ".join(_missing_or_mismatched))
        _sp.run([
            _sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            *_missing_or_mismatched,
        ], check=True)
        raise SystemExit(
            "Colab environment updated. Restart the runtime once, then rerun the notebook from the top."
        )
    else:
        print("Colab environment already compatible. No reinstall needed.")
else:
    print("Local environment detected. Skipping Colab setup.")


In [ ]:
if RUN_TARGET == "colab":
    from google.colab import drive as _drive
    from pathlib import Path

    _drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/XAllergen2.0")
    DRIVE_MODELS = DRIVE_ROOT / "models"
    DRIVE_RESULTS = DRIVE_ROOT / "results"
    print(f"Google Drive mounted: {DRIVE_ROOT}")
else:
    print("Local run: skipping Google Drive mount.")


# 07 - ICML Workshop Paper Figures and Tables

This notebook consumes supported saved probe-row artifacts from notebook 06, builds registry-aware summaries, and renders publication tables and figures under `results/paper_tables/` and `results/paper_figures/`. It intentionally excludes retired top-1-unfrozen baseline checkpoints and does not attempt to infer arbitrary model families from free-form filenames.


In [ ]:
import sys
from pathlib import Path

for _candidate in [Path.cwd(), *Path.cwd().parents]:
    _src_dir = _candidate / "src"
    if (_src_dir / "xallergen").exists() and str(_src_dir) not in sys.path:
        sys.path.insert(0, str(_src_dir))
        break
import importlib

if RUN_TARGET == "colab":
    RUNTIME_ROOT = Path("/content/XAllergen2.0")
    if str(RUNTIME_ROOT) not in sys.path:
        sys.path.insert(0, str(RUNTIME_ROOT))
    _SRC_DIR = RUNTIME_ROOT / "src"
    if _SRC_DIR.exists() and str(_SRC_DIR) not in sys.path:
        sys.path.insert(0, str(_SRC_DIR))

import xallergen.baseline_notebook_utils as baseline_notebook_utils
import xallergen.mtl_epitope_notebook_utils as mtl_epitope_notebook_utils
import xallergen.plotting_paper_figures as plotting_paper_figures

importlib.reload(baseline_notebook_utils)
importlib.reload(mtl_epitope_notebook_utils)
importlib.reload(plotting_paper_figures)

from xallergen.baseline_notebook_utils import configure_matplotlib_cache, detect_device, find_project_root, print_runtime_context
from xallergen.mtl_epitope_notebook_utils import (
    discover_supported_probe_row_artifacts,
    get_retired_checkpoint_names,
    get_supported_probe_model_registry,
    load_protein_level_metrics,
    load_supported_probe_rows,
    save_combined_probe_tables,
)
from xallergen.plotting_paper_figures import (
    ACTIVE_METHOD_KEYS,
    METHOD_PUBLICATION_LABELS,
    compute_residue_prevalence,
    plot_main_ig_masking_vs_random,
    plot_main_residue_alignment_subset,
    plot_main_saturation_mutagenesis_summary,
    plot_supplementary_all_signals_heatmap,
    plot_supplementary_label_scrambling_sanity_check,
    summarize_main_residue_alignment_subset,
    write_main_protein_performance_table,
    write_main_residue_alignment_table,
    write_supplementary_signal_tables,
)

if RUN_TARGET != "colab":
    configure_matplotlib_cache(Path.cwd())

import pandas as pd


In [ ]:
if RUN_TARGET == "colab":
    PROJECT_ROOT = RUNTIME_ROOT
    MODELS_DIR = DRIVE_MODELS if DRIVE_MODELS.exists() else PROJECT_ROOT / "models"
    RESULTS_DIR = DRIVE_RESULTS if DRIVE_RESULTS.exists() else PROJECT_ROOT / "results"
    print(f"Using models dir: {MODELS_DIR}")
    print(f"Using results dir: {RESULTS_DIR}")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    DEVICE = detect_device()
    print_runtime_context(DEVICE, PROJECT_ROOT)
    MODELS_DIR = PROJECT_ROOT / "models"
    RESULTS_DIR = PROJECT_ROOT / "results"

PAPER_FIGURES_DIR = RESULTS_DIR / "paper_figures"
PAPER_TABLES_DIR = RESULTS_DIR / "paper_tables"
PAPER_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PAPER_TABLES_DIR.mkdir(parents=True, exist_ok=True)


## Registry Discovery and Integrity Checks

Notebook 07 discovers supported probe rows from the curated registry only: Frozen ESM-2, MTL ESM-2, DeepPlantAllergy, and optional supplementary MTL ESM-2 top-1 if its saved probe rows already exist. Retired top-1-unfrozen baseline probe artifacts are intentionally excluded.


In [ ]:
retired_checkpoint_names = list(get_retired_checkpoint_names())
supported_registry = get_supported_probe_model_registry(
    models_dir=MODELS_DIR,
    results_dir=RESULTS_DIR,
    include_supplementary=True,
)
discovery_df = discover_supported_probe_row_artifacts(
    models_dir=MODELS_DIR,
    results_dir=RESULTS_DIR,
    include_supplementary=True,
)
all_probe_df = load_supported_probe_rows(discovery_df)
combined_probe_outputs = save_combined_probe_tables(all_probe_df, RESULTS_DIR)
protein_metrics_df, metrics_diagnostics_df = load_protein_level_metrics(
    results_dir=RESULTS_DIR,
    models_dir=MODELS_DIR,
    include_supplementary=False,
)
residue_prevalence = compute_residue_prevalence(all_probe_df)

active_found = discovery_df.loc[
    discovery_df["probe_rows_exists"] & ~discovery_df["supplementary_only"],
    "display_label",
].tolist()
optional_found = discovery_df.loc[
    discovery_df["probe_rows_exists"] & discovery_df["supplementary_only"],
    "display_label",
].tolist()
missing_expected = discovery_df.loc[
    ~discovery_df["probe_rows_exists"] & ~discovery_df["supplementary_only"],
    "probe_rows_path",
].astype(str).tolist()
classification_found = metrics_diagnostics_df.loc[metrics_diagnostics_df["metrics_found"], ["display_label", "metrics_path"]]
classification_missing = metrics_diagnostics_df.loc[~metrics_diagnostics_df["metrics_found"], "display_label"].tolist()
rows_per_model = all_probe_df.groupby("model_family").size().rename("n_probe_rows")
proteins_per_model = all_probe_df.groupby("model_family")["accession"].nunique().rename("n_proteins")
methods_per_model = (
    all_probe_df.groupby("model_family")["method"]
    .apply(lambda values: ", ".join(sorted(METHOD_PUBLICATION_LABELS.get(str(value), str(value)) for value in pd.unique(values))))
    .rename("available_methods")
)
label_variants = sorted(str(value) for value in all_probe_df["label_variant"].dropna().unique())
ig_validation_sweep_csv = RESULTS_DIR / "insilico_mutagenesis" / "ig_validation_sweep.csv"
ig_vs_random_csv = RESULTS_DIR / "insilico_mutagenesis" / "ig_vs_random_baseline.csv"
saturation_transition_csv = RESULTS_DIR / "insilico_mutagenesis" / "transition_panel1_by_residue_summary.csv"
per_protein_deep_dive_csv = RESULTS_DIR / "insilico_mutagenesis" / "per_protein_deep_dive.csv"
n_ig_masking_proteins = (
    int(pd.read_csv(ig_validation_sweep_csv)["sequence_id"].nunique())
    if ig_validation_sweep_csv.exists()
    else 0
)

print("Active models found:", ", ".join(active_found) if active_found else "<none>")
print("Optional models found:", ", ".join(optional_found) if optional_found else "<none>")
print("Retired models intentionally excluded:", ", ".join(retired_checkpoint_names))
print("Missing expected probe-row files:")
for path in missing_expected or ["<none>"]:
    print(f"  {path}")
print("Classification JSONs found:")
if classification_found.empty:
    print("  <none>")
else:
    for row in classification_found.itertuples(index=False):
        print(f"  {row.display_label}: {row.metrics_path}")
print("Classification JSONs missing:")
for label in classification_missing or ["<none>"]:
    print(f"  {label}")
print("Probe rows per model:")
print(rows_per_model.to_string())
print("Proteins per model:")
print(proteins_per_model.to_string())
print("Available methods per model:")
print(methods_per_model.to_string())
print("Label variants available:", ", ".join(label_variants))
print(f"Residue prevalence used as AUPRC baseline: {residue_prevalence:.4f}")
print(f"Proteins available for IG masking: {n_ig_masking_proteins}")
print(f"Saturation mutagenesis summary found: {saturation_transition_csv.exists()}")

discovery_df


## Main Paper Protein-Level Performance

This table establishes that residue-level faithfulness is being evaluated for strong protein-level classifiers rather than weak models. Frozen ESM-2, MTL ESM-2, and DeepPlantAllergy are included when their saved JSON metrics are available.


In [ ]:
MAIN_PROTEIN_CSV = PAPER_TABLES_DIR / "main_protein_level_performance.csv"
MAIN_PROTEIN_TEX = PAPER_TABLES_DIR / "main_protein_level_performance.tex"
main_protein_table_df = write_main_protein_performance_table(
    protein_metrics_df,
    MAIN_PROTEIN_CSV,
    MAIN_PROTEIN_TEX,
)
main_protein_table_df


## Main Paper Residue Alignment

The main figure and table use a curated subset of residue-level signals that best support the paper narrative: a null random baseline, post-hoc signals from Frozen ESM-2, the supervised MTL residue head, and a DeepPlantAllergy comparator when available. The full all-signal inventory is deferred to the supplementary table.


In [ ]:
MAIN_ALIGNMENT_SUMMARY_CSV = PAPER_TABLES_DIR / "main_residue_alignment_summary.csv"
MAIN_ALIGNMENT_SUMMARY_TEX = PAPER_TABLES_DIR / "main_residue_alignment_summary.tex"
MAIN_ALIGNMENT_PDF = PAPER_FIGURES_DIR / "main_residue_alignment.pdf"
MAIN_ALIGNMENT_PNG = PAPER_FIGURES_DIR / "main_residue_alignment.png"

main_alignment_summary_df, main_alignment_prevalence = summarize_main_residue_alignment_subset(all_probe_df)
main_alignment_table_df = write_main_residue_alignment_table(
    main_alignment_summary_df,
    MAIN_ALIGNMENT_SUMMARY_CSV,
    MAIN_ALIGNMENT_SUMMARY_TEX,
)
plot_main_residue_alignment_subset(
    main_alignment_summary_df,
    main_alignment_prevalence,
    MAIN_ALIGNMENT_PDF,
    MAIN_ALIGNMENT_PNG,
)
main_alignment_table_df


## IG Masking Faithfulness Figure

This figure evaluates model faithfulness for the Frozen ESM-2 classifier itself. It uses saved mutagenesis-stage CSVs only; notebook 07 does not recompute attributions or masking experiments.


In [ ]:
IG_MASKING_PDF = PAPER_FIGURES_DIR / "main_ig_masking_vs_random.pdf"
IG_MASKING_PNG = PAPER_FIGURES_DIR / "main_ig_masking_vs_random.png"
ig_masking_info = None
if ig_validation_sweep_csv.exists() and ig_vs_random_csv.exists():
    ig_masking_info = plot_main_ig_masking_vs_random(
        ig_validation_sweep_csv,
        ig_vs_random_csv,
        IG_MASKING_PDF,
        IG_MASKING_PNG,
    )
    print(ig_masking_info)
else:
    print("Missing IG masking CSV(s); skipping main_ig_masking_vs_random figure.")


## Optional Saturation Mutagenesis Summary

If the compact saturation-mutagenesis summary files are already present under `results/insilico_mutagenesis/`, this cell renders a lightweight main-paper-friendly summary. If not, it skips cleanly.


In [ ]:
SATURATION_PDF = PAPER_FIGURES_DIR / "main_saturation_mutagenesis_summary.pdf"
SATURATION_PNG = PAPER_FIGURES_DIR / "main_saturation_mutagenesis_summary.png"
saturation_generated = plot_main_saturation_mutagenesis_summary(
    per_protein_deep_dive_csv,
    saturation_transition_csv,
    SATURATION_PDF,
    SATURATION_PNG,
) if saturation_transition_csv.exists() else False
print(f"Saturation mutagenesis figure generated: {saturation_generated}")


## Supplementary All-Signals Outputs

The supplementary table keeps the full active method inventory across active and optional supported models. This includes MTL IG, MTL occlusion, attention, Grad×Input, SmoothGrad-IG, and optional MTL ESM-2 top-1 if saved probe rows already exist. Retired top-1-unfrozen baseline artifacts remain excluded.


In [ ]:
SUPPLEMENTARY_ALL_SIGNALS_CSV = PAPER_TABLES_DIR / "supplementary_all_signals_summary.csv"
SUPPLEMENTARY_ALL_SIGNALS_TEX = PAPER_TABLES_DIR / "supplementary_all_signals_summary.tex"
SUPPLEMENTARY_HEATMAP_PDF = PAPER_FIGURES_DIR / "supplementary_all_signals_heatmap.pdf"
SUPPLEMENTARY_HEATMAP_PNG = PAPER_FIGURES_DIR / "supplementary_all_signals_heatmap.png"
SUPPLEMENTARY_SCRAMBLING_PDF = PAPER_FIGURES_DIR / "supplementary_label_scrambling_sanity_check.pdf"
SUPPLEMENTARY_SCRAMBLING_PNG = PAPER_FIGURES_DIR / "supplementary_label_scrambling_sanity_check.png"

supplementary_signal_table_df = write_supplementary_signal_tables(
    all_probe_df,
    SUPPLEMENTARY_ALL_SIGNALS_CSV,
    SUPPLEMENTARY_ALL_SIGNALS_TEX,
)
supplementary_heatmap_generated = plot_supplementary_all_signals_heatmap(
    all_probe_df,
    SUPPLEMENTARY_HEATMAP_PDF,
    SUPPLEMENTARY_HEATMAP_PNG,
)
supplementary_scrambling_generated = plot_supplementary_label_scrambling_sanity_check(
    all_probe_df,
    SUPPLEMENTARY_SCRAMBLING_PDF,
    SUPPLEMENTARY_SCRAMBLING_PNG,
)
print("Supplementary LaTeX table written; if it is too wide for the final paper, split it or keep the CSV as the canonical artifact.")
print(f"Supplementary heatmap generated: {supplementary_heatmap_generated}")
print(f"Supplementary label-scrambling figure generated: {supplementary_scrambling_generated}")
supplementary_signal_table_df.head()


## Saved Outputs and LaTeX Snippets

This final cell prints the main generated paper artifacts together with the LaTeX snippets requested for the workshop draft.


In [ ]:
saved_outputs = [
    MAIN_PROTEIN_CSV,
    MAIN_PROTEIN_TEX,
    MAIN_ALIGNMENT_SUMMARY_CSV,
    MAIN_ALIGNMENT_SUMMARY_TEX,
    MAIN_ALIGNMENT_PDF,
    MAIN_ALIGNMENT_PNG,
    SUPPLEMENTARY_ALL_SIGNALS_CSV,
    SUPPLEMENTARY_ALL_SIGNALS_TEX,
]
if ig_masking_info is not None:
    saved_outputs.extend([IG_MASKING_PDF, IG_MASKING_PNG])
if saturation_generated:
    saved_outputs.extend([SATURATION_PDF, SATURATION_PNG])
if supplementary_heatmap_generated:
    saved_outputs.extend([SUPPLEMENTARY_HEATMAP_PDF, SUPPLEMENTARY_HEATMAP_PNG])
if supplementary_scrambling_generated:
    saved_outputs.extend([SUPPLEMENTARY_SCRAMBLING_PDF, SUPPLEMENTARY_SCRAMBLING_PNG])
print("Saved paper outputs:")
for path in saved_outputs:
    print(f"  {path}")

print(r"""\begin{table}[t]
  \centering
  \caption{Protein-level allergenicity classification performance. Strong protein-level performance establishes that the residue-level faithfulness analysis probes explanation quality rather than classifier failure.}
  \label{tab:protein-performance}
  \input{tables/main_protein_level_performance.tex}
\end{table}""")
print()
print(r"""\begin{figure}[t]
  \centering
  \includegraphics[width=\columnwidth]{figures/main_residue_alignment.pdf}
  \caption{Residue-level signals show limited alignment with experimentally annotated epitopes despite strong protein-level allergenicity prediction. The main panel shows a curated subset of signals: a null random baseline, post-hoc attribution/perturbation signals from the frozen ESM-2 classifier, and the supervised MTL residue head. Full all-signal results, including MTL IG and MTL occlusion, are reported in Supplementary Table~X.}
  \label{fig:residue-alignment}
\end{figure}""")
print()
print(r"""\begin{figure}[t]
  \centering
  \includegraphics[width=\columnwidth]{figures/main_ig_masking_vs_random.pdf}
  \caption{IG-highlighted residues are causally important for the frozen ESM-2 classifier's own predictions, as masking them reduces predicted allergenicity more than random masking. This evaluates model faithfulness rather than immunological epitope alignment.}
  \label{fig:ig-masking}
\end{figure}""")
print()
print(r"""\begin{table}[t]
  \centering
  \caption{Full residue-level alignment results across all active models and available importance signals.}
  \label{tab:all-signals}
  \input{tables/supplementary_all_signals_summary.tex}
\end{table}""")
